# Effective binding — OLMo

This notebook preserves the existing binding workflow but adds value/OV norms. Run one model and one prompt condition at a time.


## Setup

In [ ]:
# Cell 0: Environment Detection
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

In [ ]:
# Cell 1: Colab Only — Install pinned dependencies
# ⚠️ Restart runtime after running this cell, then skip to Cell 2
if IN_COLAB:
    %pip install -q transformer_lens==2.18.0
    %pip install -q numpy==1.26.4
    %pip install -q transformers==4.57.6

In [ ]:
# Cell 1a: Confirm Transformer Lens version
from importlib.metadata import version
print("TransformerLens version:", version("transformer-lens"))

In [ ]:
# Cell 1b: Environment check after session restart
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

In [ ]:
# Cell 2: Project Root & Path Setup
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("GH_TMLR")

    repo_owner = "trishasalas"
    repo_name = "tmlr"
    repo_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"

    PROJECT_ROOT = Path("/content") / repo_name

    if not PROJECT_ROOT.exists():
        !git clone {repo_url} {PROJECT_ROOT}
else:
    # Local: notebook lives in notebooks/, project root is one level up
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Cell 3: Imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added {PROJECT_ROOT} to sys.path")

import torch
import src
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

# Device selection
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

In [ ]:
import src
from src.olmo_config import OLMO_REVISIONS
from src.tl217_olmo2_adapter import load_olmo2_tl217

In [ ]:
# Cell 5 - Model name variable
model_name = "OLMo-2-0425-1B"
revision = OLMO_REVISIONS[model_name]

In [ ]:
# Cell 6: Load Model
model = load_olmo2_tl217(f"allenai/{model_name}", device=device, revision=revision)

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
FAMILY_NAME = "olmo"
print("Family:", FAMILY_NAME)


## Norm-aware binding run

Start with `PROMPT_CONDITION = "natural"`. After it finishes, change that one setting to `"uniform"` and rerun this cell. The two runs are saved separately and cannot overwrite each other.

The notebook defaults to the 49 accessibility compounds because those are the compounds with the frequency table used in the primary test.

For OLMo, `weighted_ov_norm` is the source-specific write immediately before OLMo's attention-branch RMS normalization. The notebook labels this explicitly so it is not overstated as a complete ALTI-style information-flow measure.


In [ ]:
# Effective Binding Battery
#
# This is the existing binding sweep plus norm-aware measurements. It records:
#   binding_score                  raw attention (kept for compatibility)
#   ov_write_norm                 size of the source token's OV write
#   weighted_ov_norm              attention × OV-write size
#   relative_weighted_ov_norm     weighted write / target residual size
#
# Start with accessibility only. Change PROMPT_CONDITION and rerun the notebook
# to create the uniform-template replication in a separate folder.
from pathlib import Path
import math
import torch
import yaml
import pandas as pd
from src.effective_binding_manifest import write_effective_binding_manifest

PROMPT_CONDITION = "natural"       # allowed: "natural" or "uniform"
DOMAINS_TO_RUN = ["accessibility"] # primary 49-compound analysis

assert PROMPT_CONDITION in {"natural", "uniform"}


def find_token_index(tokens, target):
    """Return the LAST token index belonging to target, including split words."""
    target_lower = target.lower()

    for i, tok in enumerate(tokens):
        if tok.strip().lower() == target_lower:
            return i

    for start in range(len(tokens)):
        joined = ""
        for end in range(start, len(tokens)):
            joined += tokens[end].strip().lower()
            if joined == target_lower:
                return end
            if len(joined) > len(target_lower):
                break
    return None


def make_prompt(case, condition):
    if condition == "natural":
        return case["prompt"]
    return f"A {case['word1']} {case['word2']} is"


family = FAMILY_NAME
output_dir = (
    PROJECT_ROOT / "results" / "effective_binding" /
    family / model_name / PROMPT_CONDITION
)
output_dir.mkdir(parents=True, exist_ok=True)

all_results = []
unresolved = []
domain_counts = {}
prompt_files_used = []

for domain in DOMAINS_TO_RUN:
    prompts_path = PROJECT_ROOT / "data" / "binding" / f"{domain}.yaml"
    prompt_files_used.append(prompts_path)
    with open(prompts_path, "r") as f:
        compounds = yaml.safe_load(f)["compounds"]

    print(f"\n--- Running {domain}: {len(compounds)} compounds ---")
    results = []

    for i, case in enumerate(compounds):
        print(f"\r  {i + 1}/{len(compounds)}  {case['name']:<30}", end="")
        name = case["name"]
        word1, word2 = case["word1"], case["word2"]
        prompt = make_prompt(case, PROMPT_CONDITION)

        tokens = model.to_str_tokens(prompt)
        idx1 = find_token_index(tokens, word1)
        idx2 = find_token_index(tokens, word2)

        if idx1 is None or idx2 is None:
            unresolved.append((domain, name, word1, word2, tokens))
            continue

        source_idx = min(idx1, idx2)
        target_idx = max(idx1, idx2)

        # We cache only the three activations needed by this experiment.
        def needed(name):
            return (
                name.endswith("hook_pattern")
                or name.endswith("hook_v")
                or name.endswith("hook_resid_pre")
            )

        with torch.no_grad():
            _, cache = model.run_with_cache(prompt, names_filter=needed)

            for layer in range(model.cfg.n_layers):
                # [heads]
                attention = cache["pattern", layer][0, :, target_idx, source_idx]

                # Source token's value vector, then each head's OV write:
                # [heads, d_head] @ [heads, d_head, d_model]
                source_v = cache["v", layer][0, source_idx]
                ov_write = torch.einsum(
                    "hd,hdm->hm", source_v, model.W_O[layer]
                )
                ov_write_norm = ov_write.float().norm(dim=-1)
                weighted_ov_norm = attention.float() * ov_write_norm

                target_residual_norm = (
                    cache["resid_pre", layer][0, target_idx]
                    .float()
                    .norm()
                    .clamp_min(1e-12)
                )
                relative = weighted_ov_norm / target_residual_norm

                for head in range(model.cfg.n_heads):
                    results.append({
                        "compound": name,
                        "layer": layer,
                        "head": head,
                        # Compatibility with the original binding files:
                        "binding_score": attention[head].item(),
                        "attention_weight": attention[head].item(),
                        "ov_write_norm": ov_write_norm[head].item(),
                        "weighted_ov_norm": weighted_ov_norm[head].item(),
                        "relative_weighted_ov_norm": relative[head].item(),
                        "target_residual_norm": target_residual_norm.item(),
                        "word1": word1,
                        "word2": word2,
                        "prompt": prompt,
                        "prompt_condition": PROMPT_CONDITION,
                        "tokens": str(tokens),
                        "word1_idx": source_idx,
                        "word2_idx": target_idx,
                        "domain": domain,
                        "family": family,
                        "model": model_name,
                    })

        del cache

    print()
    domain_df = pd.DataFrame(results)
    filename = f"{model_name}-{PROMPT_CONDITION}-{domain}.csv"
    domain_df.to_csv(output_dir / filename, index=False)
    all_results.append(domain_df)
    domain_counts[domain] = {
        "expected": len(compounds) * model.cfg.n_layers * model.cfg.n_heads,
        "written": len(domain_df),
        "file": filename,
    }
    print(f"  → {filename}  ({len(domain_df):,} rows)")

results_df = pd.concat(all_results, ignore_index=True)

expected = sum(item["expected"] for item in domain_counts.values())

print(f"\nExpected rows: {expected:,}")
print(f"Written rows:  {len(results_df):,}")
print(f"Missing rows:  {expected - len(results_df):,}")
print(f"Output folder: {output_dir}")

if unresolved:
    print(f"\n⚠️ {len(unresolved)} unresolved compounds:")
    for domain, name, word1, word2, tokens in unresolved:
        print(f"  {domain}/{name}: {word1!r}, {word2!r} → {tokens}")
else:
    print("\n✓ Every compound was resolved into token positions.")

# A visual sanity check. These values must be finite and non-negative.
measurement_columns = [
    "attention_weight", "ov_write_norm", "weighted_ov_norm",
    "relative_weighted_ov_norm", "target_residual_norm",
]
assert results_df[measurement_columns].notna().all().all()
assert (results_df[measurement_columns] >= 0).all().all()

manifest_path = write_effective_binding_manifest(
    project_root=PROJECT_ROOT,
    output_dir=output_dir,
    model_name=model_name,
    model=model,
    prompt_condition=PROMPT_CONDITION,
    family=family,
    domain_counts=domain_counts,
    results_df=results_df,
    prompt_files=prompt_files_used,
    unresolved=unresolved,
    revision=globals().get("revision"),
    hf_commit_sha=globals().get("hf_commit_sha"),
)

display(results_df.head())


In [ ]:
import os
os.chdir(PROJECT_ROOT)
!git config user.email "trisha@trishasalas.com"
!git config user.name "Trisha Salas"
!git add results/effective_binding/
!git commit -m "effective binding: {model_name} {PROMPT_CONDITION}"
!git push


### Delete Model & Clear Cache

In [ ]:
# Cell 7: Free memory for next model
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")